In [0]:
%run /Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config


Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# — Silver | Clean and normalize YouTube trailer data
from pyspark.sql import functions as F

# Load Bronze table
df_youtube = spark.table(f"{BRONZE_PATH}.youtube_trailers_raw")

# Fix data types
df_youtube = (df_youtube
    .withColumn("view_count",    F.col("view_count").cast("long"))
    .withColumn("like_count",    F.col("like_count").cast("long"))
    .withColumn("comment_count", F.col("comment_count").cast("long"))
    .withColumn("published_at",  F.to_timestamp(F.col("published_at")))
    .withColumn("release_date",  F.to_date(F.col("release_date")))
)

# Days between trailer drop and release date
df_youtube = df_youtube.withColumn(
    "days_trailer_to_release",
    F.datediff(F.col("release_date"), F.col("published_at").cast("date"))
)

# Normalize view count to 0-100 scale across all films
max_views = df_youtube.agg(F.max("view_count")).collect()[0][0]
df_youtube = df_youtube.withColumn(
    "view_count_norm",
    (F.col("view_count") / max_views * 100).cast("double")
)

# Engagement rate = (likes + comments) / views
df_youtube = df_youtube.withColumn(
    "engagement_rate",
    ((F.col("like_count") + F.col("comment_count")) / 
     F.col("view_count") * 100).cast("double")
)

# Assign era
df_youtube_silver = df_youtube.withColumn("era",
    F.when(F.year(F.col("release_date")) <= 2014, "Pre Short-Form")
     .when(F.year(F.col("release_date")) <= 2018, "Transition")
     .otherwise("Short-Form Era")
)

# Select final columns
df_youtube_silver = df_youtube_silver.select(
    "film", "release_date", "genre", "era",
    "view_count", "view_count_norm",
    "like_count", "comment_count",
    "engagement_rate", "days_trailer_to_release"
)

print(f"✅ YouTube silver ready — {df_youtube_silver.count()} films")
display(df_youtube_silver)

✅ YouTube silver ready — 50 films


film,release_date,genre,era,view_count,view_count_norm,like_count,comment_count,engagement_rate,days_trailer_to_release
Batman Begins,2005-06-15,Action,Pre Short-Form,6244470,7.166475647717892,49837,2606,0.839831082541833,-3102
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form,814753,0.9350525398320587,3673,498,0.5119342917424055,-2195
Casino Royale,2006-11-17,Action,Pre Short-Form,5596494,6.422824989726797,27915,1492,0.5254539717187224,-2083
The Departed,2006-10-06,Drama,Pre Short-Form,5249806,6.024947970643349,27018,1686,0.5467630613397905,-2779
No Country for Old Men,2007-11-09,Thriller,Pre Short-Form,5786136,6.640467924160707,34339,1513,0.6196190341879279,-2374
There Will Be Blood,2007-12-26,Drama,Pre Short-Form,3958323,4.542775509418994,24781,1562,0.6655091057500866,-2327
Ratatouille,2007-06-29,Comedy,Pre Short-Form,7314038,8.393967016173233,18814,0,0.25723136795296936,-3785
The Dark Knight,2008-07-18,Action,Pre Short-Form,21476936,24.64803879778359,208517,13740,1.034863632317012,-1964
Inception,2010-07-16,Action,Pre Short-Form,20144125,23.118436193477613,169626,6418,0.8739222974440439,-1236
Avatar,2009-12-18,Sci-Fi,Pre Short-Form,43758842,50.21990266032744,669551,37568,1.615945412815083,-5701


In [0]:
# — Save to Silver Delta table
(df_youtube_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER_PATH}.youtube_silver")
)

print(f"✅ Saved to {SILVER_PATH}.youtube_silver")
print(f"   Rows written: {df_youtube_silver.count()}")

display(spark.table(f"{SILVER_PATH}.youtube_silver"))

✅ Saved to bootcamp_students.tiffani_ceresrain_silver.youtube_silver
   Rows written: 50


film,release_date,genre,era,view_count,view_count_norm,like_count,comment_count,engagement_rate,days_trailer_to_release
Batman Begins,2005-06-15,Action,Pre Short-Form,6244470,7.166475647717892,49837,2606,0.839831082541833,-3102
Brokeback Mountain,2005-12-09,Drama,Pre Short-Form,814753,0.9350525398320587,3673,498,0.5119342917424055,-2195
Casino Royale,2006-11-17,Action,Pre Short-Form,5596494,6.422824989726797,27915,1492,0.5254539717187224,-2083
The Departed,2006-10-06,Drama,Pre Short-Form,5249806,6.024947970643349,27018,1686,0.5467630613397905,-2779
No Country for Old Men,2007-11-09,Thriller,Pre Short-Form,5786136,6.640467924160707,34339,1513,0.6196190341879279,-2374
There Will Be Blood,2007-12-26,Drama,Pre Short-Form,3958323,4.542775509418994,24781,1562,0.6655091057500866,-2327
Ratatouille,2007-06-29,Comedy,Pre Short-Form,7314038,8.393967016173233,18814,0,0.25723136795296936,-3785
The Dark Knight,2008-07-18,Action,Pre Short-Form,21476936,24.64803879778359,208517,13740,1.034863632317012,-1964
Inception,2010-07-16,Action,Pre Short-Form,20144125,23.118436193477613,169626,6418,0.8739222974440439,-1236
Avatar,2009-12-18,Sci-Fi,Pre Short-Form,43758842,50.21990266032744,669551,37568,1.615945412815083,-5701
